In [4]:
import psycopg2
import csv
from datetime import datetime, timedelta

# Параметры подключения к базе данных
DB_PARAMS = {
    'dbname': 'forum_db',
    'user': 'user',
    'password': 'password',  # Замените на свой пароль
    'host': 'localhost',
    'port': '5432'
}

# Функция для подключения к базе данных
def connect_db():
    return psycopg2.connect(**DB_PARAMS)

# Функция для получения агрегированных данных
def aggregate_data(conn, start_date, end_date):
    result = []
    
    # Количество новых аккаунтов
    with conn.cursor() as cur:
        cur.execute("""
            SELECT COUNT(*) 
            FROM users
            WHERE registration_date BETWEEN %s AND %s
        """, (start_date, end_date))
        new_users = cur.fetchone()[0]
    
    # Количество сообщений всего и количество сообщений анонимов
    with conn.cursor() as cur:
        cur.execute("""
            SELECT COUNT(*) 
            FROM messages
            WHERE created_at BETWEEN %s AND %s
        """, (start_date, end_date))
        total_messages = cur.fetchone()[0]
        
        cur.execute("""
            SELECT COUNT(*) 
            FROM messages
            WHERE created_at BETWEEN %s AND %s AND anon_id IS NOT NULL
        """, (start_date, end_date))
        anonymous_messages = cur.fetchone()[0]
    
    # Прирост количества тем относительно предыдущего дня
    with conn.cursor() as cur:
        cur.execute("""
            SELECT COUNT(*) 
            FROM topics
            WHERE created_at BETWEEN %s AND %s
        """, (start_date, end_date))
        topics_today = cur.fetchone()[0]
        
        prev_day = start_date - timedelta(days=1)
        cur.execute("""
            SELECT COUNT(*) 
            FROM topics
            WHERE created_at BETWEEN %s AND %s
        """, (prev_day, start_date))
        topics_yesterday = cur.fetchone()[0]
        
        if topics_yesterday > 0:
            topics_growth = (topics_today - topics_yesterday) / topics_yesterday * 100
        else:
            topics_growth = 0
    
    # Собираем все данные
    result.append([
        start_date.date(),
        new_users,
        anonymous_messages / total_messages * 100 if total_messages > 0 else 0,
        total_messages,
        topics_growth
    ])
    
    return result

# Функция для записи данных в CSV
def write_to_csv(data):
    filename = 'aggregated_data.csv'
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Day', 'New Accounts', 'Anonymous Messages (%)', 'Total Messages', 'Topics Growth (%)'])
        for row in data:
            writer.writerow(row)
    print(f"Данные успешно записаны в {filename}")

# Основная функция
def main():
    conn = connect_db()
    end_date = datetime.now()
    start_date = end_date - timedelta(days=30)  # Агрегируем данные за последние 30 дней
    
    aggregated_data = []
    current_date = start_date
    
    while current_date <= end_date:
        next_day = current_date + timedelta(days=1)
        data = aggregate_data(conn, current_date, next_day)
        aggregated_data.extend(data)
        current_date = next_day
    
    write_to_csv(aggregated_data)
    conn.close()

if __name__ == "__main__":
    main()


OperationalError: connection to server at "localhost" (::1), port 5432 failed: FATAL:  database "forum_db" does not exist
